# 🛡️ Ejercicios: PayPal
# Fraude y disputas por tipo, país e importe

⚠️ Datos SIMULADOS / ilustrativos
(generados con pandas + numpy).
No son cifras oficiales de PayPal.

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué país concentra
### más fraude y cuánto
### mueve en €?

No basta el % de fraude: hay que ver
cuántos € están en juego por país.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/paypal_fraude.csv'
)

r = (
  df
  .groupby('pais')
  .agg(
    tasa=('es_fraude', 'mean'),
    eur=('importe_eur', 'sum'),
    txs=('es_fraude', 'size'),
  )
)

r['tasa_%'] = (r['tasa'] * 100).round(1)
r = r.drop(columns='tasa')

print('Fraude por país:')
print(
  r.sort_values('tasa_%',
    ascending=False).round(0)
)

### 💼 PREGUNTA ENTREVISTA:
### ¿Los importes altos
### son más propensos
### al fraude? (bins)

Partimos el importe en tramos y
medimos la tasa de fraude en cada uno.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/paypal_fraude.csv'
)

df['tramo'] = pd.cut(
  df['importe_eur'],
  bins=[0, 25, 75, 150, 1e9],
  labels=['0-25', '25-75',
          '75-150', '150+'],
)

r = (
  df
  .groupby('tramo', observed=True)
  ['es_fraude']
  .mean()
  .mul(100)
  .round(1)
)

print('Tasa fraude % por tramo €:')
print(r)

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué % de fraudes
### acaba en disputa
### y se resuelve?

Embudo: fraude → disputa → resuelta.
Mide la fuga de dinero no recuperado.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/paypal_fraude.csv'
)

f = df[df['es_fraude'] == 1]

n = len(f)
disp = f['disputa'].sum()
res = f['resuelta'].sum()

print(f'Fraudes: {n}')
print(
  f'Con disputa: {disp} '
  f'({disp/n*100:.1f}%)'
)
print(
  f'Resueltas: {res} '
  f'({res/disp*100:.1f}% disp.)'
)

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué tipo de pago
### es más arriesgado?
### (fraude vs ticket)

Cruzamos tasa de fraude e importe
medio por tipo para priorizar reglas.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/paypal_fraude.csv'
)

r = (
  df
  .groupby('tipo')
  .agg(
    fraude=('es_fraude', 'mean'),
    ticket=('importe_eur', 'mean'),
  )
)

r['fraude_%'] = (
  r['fraude'] * 100
).round(1)
r['ticket'] = r['ticket'].round(1)
r = r.drop(columns='fraude')

print('Riesgo por tipo de pago:')
print(
  r.sort_values('fraude_%',
    ascending=False)
)

### 💼 PREGUNTA ENTREVISTA:
### ¿Cuánto € en fraude
### NO recuperamos?
### (pérdida neta)

Fraude sin disputa resuelta a favor
= dinero perdido. Lo cuantificamos.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/paypal_fraude.csv'
)

f = df[df['es_fraude'] == 1].copy()

f['perdido'] = f['importe_eur'] * (
  f['resuelta'] == 0
)

total = f['importe_eur'].sum()
perd = f['perdido'].sum()

print(f'Fraude total: {total:,.0f} €')
print(f'No recuperado: {perd:,.0f} €')
print(
  f'% fuga: {perd/total*100:.1f}%'
)